# V2 空间生成：显式局部运输路径

首轮只修改 XY，复用已完成 G zero 的表达、类型、来源和目标深度。source_xy 对照保留来源 XY 及同一份 G zero 表达；对比 OT 重心路径（ot_barycentric）与每个来源固定抽取一个 OT 目标的路径（ot_sampled）。无需重新训练网络，也不调用原表达细化算子。

候选半径、成本尺度、UOT 参数全部来自原 G 配置。未观察到可信候选时保留来源位置。留出切片在读取前排除，不用隐藏真值调参。

这是用来检验空间运动机制的 V2 候选；是否优于来源复制，需要运行后的三折评价。

In [ ]:
from pathlib import Path
import sys
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "deepspatial/core.py").is_file() or (p / "HM_DeepSpital/deepspatial/core.py").is_file())
if (PROJECT / "HM_DeepSpital").is_dir():
    PROJECT = PROJECT / "HM_DeepSpital"
sys.path.insert(0, str(PROJECT))
from examples.run_v2_spatial import run_experiment


In [ ]:
SOURCE_ROOT = PROJECT / "outputs/g_holdouts_1_2_3"
OUTPUT_ROOT = PROJECT / "outputs/v2_spatial_holdouts_1_2_3"
HOLDOUTS = [1, 2, 3]
MODES = ["source_xy", "ot_barycentric", "ot_sampled"]
SEED = 42


In [ ]:
completed = run_experiment(
    source_root=SOURCE_ROOT, output_root=OUTPUT_ROOT,
    holdouts=HOLDOUTS, modes=MODES, seed=SEED,
)
completed


每折输出 shared/coupling_*.npz 和逐端点匹配诊断；每种空间路径输出 generated.h5ad、配置、薄层预算和逐类位移。输入文件不修改，已有输出不覆盖。

评价继续使用可见训练丰度≤5%、Z±5µm、同类25/50/100µm覆盖，同时比较空间表达PCC和密度误差。必须同时比较 G zero、同调度来源复制，以及冻结G zero表达的来源XY对照，区分位置与表达贡献。